# Assignment

## Brief

Write the Python codes for the following questions.

## Instructions

Paste the answer as Python in the answer code section below each question.

### Question 1

Question: Implement a simple Thrift server and client that defines a `Student` struct with fields `name` (string), `age` (integer), and `courses` (list of strings). Include a service `School` with a method `enrollCourse` that takes a `Student` record and a course name, adds the course to the student's course list, and returns the updated `Student` record.

Answer:

**Thrift schema (student.thrift)**

In [2]:
%%writefile student.thrift

struct Student {
  1: required string name,
  2: optional i32 age,
  3: optional list<string> courses
}

service School {
    Student enrollStudent(1: required Student student, 2: required string course)
}


Writing student.thrift


**Thrift server (student_server.py)**

In [1]:
%%writefile student_thrift_server.py
import thriftpy2
student_thrift = thriftpy2.load("./student.thrift", module_name="student_thrift")

from thriftpy2.rpc import make_server

class School(object):
    def enrollStudent(self, person, course):
        person.courses.append(course)
        return person

server = make_server(student_thrift.School, School(), client_timeout=None)
print("School (Server started)...")
print("Press Ctrl+C to stop the server.")
server.serve()

Writing student_thrift_server.py


Run the thrift server

**Thrift client (student_client.py)**

In [6]:
# Enter your student_client code here

import thriftpy2
student_thrift = thriftpy2.load("./student.thrift", module_name="student_thrift")

from thriftpy2.rpc import make_client

# Connect to the kitchen
school = make_client(student_thrift.School, timeout=None)

martin = student_thrift.Student(
    name="Martin", age=17, courses=["science", "math"]
)
print(f"Initial courses: {martin.courses}")

# Enroll Martin in a new course
martin = school.enrollStudent(martin, "history")
print(f"Courses after enrollment: {martin.courses}")

Initial courses: ['science', 'math']
Courses after enrollment: ['science', 'math', 'history']


### Question 2

Question: Implement a simple Protocol Buffers server and client that defines a `Book` message with fields `title` (string), `author` (string), and `page_count` (integer). Include a service `Library` with a method `checkoutBook` that takes a `Book` message and returns the same `Book` message.

Answer:

**Protobuf schema (book.proto)**

In [7]:
%%writefile book.proto
syntax = "proto3";

message Book {
  string title = 1;
  string author = 2;
  optional int32 page_count = 3;
}

message BookRequest {
  Book book = 1;
  string title = 2;
}

service Library {
  rpc checkoutBook(BookRequest) returns (Book) {}
}

Writing book.proto


Run

**Protobuf server (book_server.py)**

In [9]:
%%writefile book_server.py
from concurrent import futures
import grpc
import book_pb2_grpc


class Library(book_pb2_grpc.LibraryServicer):

  def checkoutBook(self, request, context):
    # Implementation for checking out a book
    return request.book

server = grpc.server(futures.ThreadPoolExecutor(max_workers=2))
book_pb2_grpc.add_LibraryServicer_to_server(
    Library(), server)
server.add_insecure_port('[::]:50051')
print("Librarian is in the Library (Server started)...")
print("Press Ctrl+C to stop the server.")
server.start()
server.wait_for_termination()


Overwriting book_server.py


Run

**Protobuf client (book_client.py)**

In [12]:
# Enter your book_client code here
import sys
sys.path.append('..') # This helps Python find the generated protobuf files
import grpc
import book_pb2
import book_pb2_grpc

def checkout_book(stub, book, title):
    # Create a BookRequest package with the book and the title
    request = book_pb2.BookRequest(book=book, title=title)
    # Send the request to the server and get the updated book back
    updated_book = stub.checkoutBook(request)
    return updated_book


# Connect to the kitchen (server) on localhost:50051
with grpc.insecure_channel('localhost:50051') as channel:
    # Create a book1 title "Alice in Wonderland" by "Lewis Carroll" with 200 pages
    book1 = book_pb2.Book(
        title="Alice in Wonderland",
        author="Lewis Carroll",
        page_count=200
    )
    print(f"Book 1: {book1.title} by {book1.author}")
    # Create a 'stub' which is like our direct line to the Library service
    stub = book_pb2_grpc.LibraryStub(channel)

    # Send the book to the library to check it out
    checked_out_book = checkout_book(stub, book1, "Alice in Wonderland")

    # Let's see the updated book
    print(f"Checked out book: {checked_out_book.title} by {checked_out_book.author}")


Book 1: Alice in Wonderland by Lewis Carroll
Checked out book: Alice in Wonderland by Lewis Carroll
